# Music Genre Classification v2 — CNN on FMA-Medium

**v2 changes from v1:** Switched from GTZAN (1K tracks, 10 genres) to **FMA-Medium (25K tracks, 16 genres)**. Uses FMA's **official train/val/test split** so the model is evaluated on tracks it has truly never seen. See `MIGRATION_V1_TO_V2.md` for full rationale.

**Steps:**
1. Install dependencies & verify GPU
2. Download FMA-Medium dataset (~22 GB audio + ~340 MB metadata)
3. Parse metadata & split into train/val/test
4. Preprocess audio → Mel spectrograms
5. Build & train CNN with class weights
6. Evaluate on held-out test set (chunk-level + file-level majority voting)
7. Inference with top-3 genre output (supports MP3/WAV)

**Runtime:** Go to `Runtime → Change runtime type → GPU` (T4 or better). FMA-Medium download takes ~20-30 min on Colab.

**Disk:** Reserve ~30 GB free disk space (raw audio + extracted spectrograms).

## 1. Setup

In [ ]:
# Install dependencies (TensorFlow is pre-installed on Colab)
!pip install -q librosa>=0.10.0 numpy pandas>=2.0 matplotlib>=3.7 seaborn>=0.12 scikit-learn>=1.3 soundfile>=0.12 tqdm
!apt-get -qq install -y ffmpeg  # needed for MP3 decoding

# Restart runtime to pick up updated numpy (Colab pre-installed packages need numpy 2.x)
import os
os.kill(os.getpid(), 9)

In [ ]:
# After runtime restart, run from here
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")
if not tf.config.list_physical_devices('GPU'):
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU")

In [ ]:
# All imports
import os
import json
import shutil
import zipfile
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Configuration

Switch `SUBSET` to `"small"` if you want a faster run (8K tracks, 8 genres, ~7 GB download).

In [ ]:
# --- Dataset selection ---
SUBSET = "medium"   # "small" (8K, 8 genres) or "medium" (25K, 16 genres)

# --- Paths ---
BASE_DIR = os.getcwd()
FMA_DIR = os.path.join(BASE_DIR, "fma_data")
AUDIO_DIR = os.path.join(FMA_DIR, f"fma_{SUBSET}")
METADATA_DIR = os.path.join(FMA_DIR, "fma_metadata")
SPECTROGRAM_DIR = os.path.join(BASE_DIR, "spectrograms")
MODEL_DIR = os.path.join(BASE_DIR, "models")
MODEL_PATH = os.path.join(MODEL_DIR, "genre_cnn_v2.keras")
HISTORY_PATH = os.path.join(MODEL_DIR, "history_v2.json")
GENRES_PATH = os.path.join(MODEL_DIR, "genres_v2.json")
PLOTS_DIR = os.path.join(BASE_DIR, "plots")

# --- Audio ---
SAMPLE_RATE = 22050
CHUNK_DURATION = 4   # seconds
CHUNKS_PER_TRACK = 7  # non-overlapping 4s chunks across the 30s clip
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_DURATION   # 88200

# --- Mel Spectrogram ---
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 2048
IMG_HEIGHT = 150
IMG_WIDTH = 150

# --- Model ---
CONV_FILTERS = [32, 64, 128, 256, 512]
KERNEL_SIZE = (3, 3)
POOL_SIZE = (2, 2)
CONV_DROPOUT = 0.30
DENSE_UNITS = 1024
DENSE_DROPOUT = 0.50

# --- Training ---
EPOCHS = 30
BATCH_SIZE = 64
LEARNING_RATE = 0.001
RANDOM_SEED = 42

# Create output directories
for d in [FMA_DIR, SPECTROGRAM_DIR, MODEL_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Using FMA-{SUBSET}")
print(f"Base directory: {BASE_DIR}")

## 3. Download FMA Dataset

Downloads from the official FMA host. This is the slow step — for FMA-Medium expect ~20-30 min on Colab.

If the download is too slow, you can:
1. Switch `SUBSET = "small"` in the config cell above
2. Or upload `fma_medium.zip` and `fma_metadata.zip` to your Google Drive and mount it

In [ ]:
# Download URLs (official FMA host)
FMA_HOST = "https://os.unil.cloud.switch.ch/fma"
URLS = {
    "metadata": f"{FMA_HOST}/fma_metadata.zip",      # ~340 MB
    "audio":    f"{FMA_HOST}/fma_{SUBSET}.zip",      # small ~7 GB, medium ~22 GB
}


def download_and_extract(url, dest_zip, extract_to):
    """Download a zip file (with progress) and extract it."""
    if not os.path.exists(dest_zip):
        print(f"Downloading {url}...")
        # Use wget with progress bar (works on Colab/Linux)
        result = os.system(f"wget -q --show-progress -O {dest_zip} {url}")
        if result != 0 or not os.path.exists(dest_zip):
            raise RuntimeError(f"Download failed: {url}")
    else:
        print(f"Already downloaded: {dest_zip}")

    print(f"Extracting to {extract_to}...")
    with zipfile.ZipFile(dest_zip, "r") as zf:
        zf.extractall(extract_to)
    print("Done.")


# Download metadata (small, fast)
if not os.path.isdir(METADATA_DIR):
    download_and_extract(
        URLS["metadata"],
        os.path.join(FMA_DIR, "fma_metadata.zip"),
        FMA_DIR,
    )
else:
    print(f"Metadata already extracted: {METADATA_DIR}")

# Download audio (large, slow)
if not os.path.isdir(AUDIO_DIR):
    download_and_extract(
        URLS["audio"],
        os.path.join(FMA_DIR, f"fma_{SUBSET}.zip"),
        FMA_DIR,
    )
else:
    print(f"Audio already extracted: {AUDIO_DIR}")

print("\nDownload + extract complete.")

## 4. Parse Metadata & Splits

FMA provides official train/validation/test splits in `tracks.csv`. We'll use these directly so our test set is genuinely held out.

In [ ]:
# Load tracks metadata (multi-level header)
tracks_path = os.path.join(METADATA_DIR, "tracks.csv")
tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1])

# Filter to the chosen subset (small is a strict subset of medium)
if SUBSET == "small":
    allowed_subsets = ["small"]
else:  # medium
    allowed_subsets = ["small", "medium"]

subset_mask = tracks[("set", "subset")].isin(allowed_subsets)
tracks_subset = tracks[subset_mask]

# Get top genre and split for each track
metadata = pd.DataFrame({
    "track_id": tracks_subset.index,
    "genre": tracks_subset[("track", "genre_top")].values,
    "split": tracks_subset[("set", "split")].values,
})
metadata = metadata.dropna(subset=["genre"]).reset_index(drop=True)

# Build the genre list (sorted alphabetically for reproducibility)
GENRES = sorted(metadata["genre"].unique().tolist())
NUM_CLASSES = len(GENRES)
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}

metadata["label"] = metadata["genre"].map(GENRE_TO_IDX)

print(f"Total tracks in fma_{SUBSET}: {len(metadata)}")
print(f"Genres ({NUM_CLASSES}): {GENRES}")
print("\nTracks per split:")
print(metadata["split"].value_counts())
print("\nTracks per genre (across all splits):")
print(metadata["genre"].value_counts())

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(12, 5))
split_counts = metadata.groupby(["genre", "split"]).size().unstack(fill_value=0)
split_counts = split_counts.reindex(GENRES)
split_counts[["training", "validation", "test"]].plot(
    kind="bar", stacked=True, ax=ax,
    color=["steelblue", "orange", "firebrick"],
)
ax.set_title(f"FMA-{SUBSET}: Tracks per Genre by Split")
ax.set_xlabel("Genre")
ax.set_ylabel("Number of tracks")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Preprocessing — Audio to Mel Spectrograms

We extract `CHUNKS_PER_TRACK` non-overlapping 4-second chunks from each 30-second clip, compute Mel spectrograms, resize to 150×150, and normalize to [0, 1]. Saved as **float16** to fit in memory.

This is the slow step — expect ~30-60 min for FMA-Medium. We process per split and save separate files.

In [ ]:
def get_audio_path(track_id, audio_dir=AUDIO_DIR):
    """e.g., 12345 -> 'fma_medium/012/012345.mp3'"""
    tid_str = f"{track_id:06d}"
    return os.path.join(audio_dir, tid_str[:3], tid_str + ".mp3")


def load_and_chunk(file_path, sr=SAMPLE_RATE,
                   chunk_samples=CHUNK_SAMPLES,
                   max_chunks=CHUNKS_PER_TRACK):
    """Load audio and split into non-overlapping fixed-length chunks."""
    audio, _ = librosa.load(file_path, sr=sr, res_type="kaiser_fast")
    chunks = []
    for start in range(0, len(audio) - chunk_samples + 1, chunk_samples):
        chunks.append(audio[start : start + chunk_samples])
        if len(chunks) >= max_chunks:
            break
    return chunks


def audio_to_mel_spectrogram(audio_chunk):
    """Compute Mel spectrogram and convert to dB."""
    mel_spec = librosa.feature.melspectrogram(
        y=audio_chunk, sr=SAMPLE_RATE, n_mels=N_MELS,
        hop_length=HOP_LENGTH, n_fft=N_FFT,
    )
    return librosa.power_to_db(mel_spec, ref=np.max)


def resize_spectrogram(spec):
    """Resize to 150x150 and normalize to [0, 1]."""
    img = Image.fromarray(spec)
    img = img.resize((IMG_WIDTH, IMG_HEIGHT), Image.LANCZOS)
    resized = np.array(img, dtype=np.float64)
    spec_min, spec_max = resized.min(), resized.max()
    if spec_max - spec_min > 0:
        resized = (resized - spec_min) / (spec_max - spec_min)
    else:
        resized = np.zeros_like(resized)
    return resized.astype(np.float16)  # float16 for memory efficiency


def preprocess_single_file(file_path, max_chunks=None):
    """Preprocess a single audio file for inference. Returns (num_chunks, 150, 150, 1)."""
    chunks = load_and_chunk(file_path, max_chunks=max_chunks or 999999)
    spectrograms = []
    for chunk in chunks:
        mel_spec = audio_to_mel_spectrogram(chunk)
        spectrograms.append(resize_spectrogram(mel_spec).astype(np.float32))
    X = np.array(spectrograms, dtype=np.float32)
    return X[..., np.newaxis]

In [ ]:
def preprocess_split(split_name, metadata):
    """Preprocess one split (training/validation/test) and save spectrograms + chunk-to-track map."""
    out_X = os.path.join(SPECTROGRAM_DIR, f"X_{split_name}.npy")
    out_y = os.path.join(SPECTROGRAM_DIR, f"y_{split_name}.npy")
    out_map = os.path.join(SPECTROGRAM_DIR, f"map_{split_name}.csv")

    if all(os.path.exists(p) for p in [out_X, out_y, out_map]):
        print(f"[{split_name}] already preprocessed, skipping.")
        return

    df = metadata[metadata["split"] == split_name]
    print(f"[{split_name}] {len(df)} tracks to preprocess...")

    all_specs = []
    all_labels = []
    map_rows = []
    skipped = 0
    chunk_idx = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc=split_name):
        track_id = row["track_id"]
        label = row["label"]
        path = get_audio_path(track_id)

        if not os.path.exists(path):
            skipped += 1
            continue

        try:
            chunks = load_and_chunk(path)
        except Exception:
            skipped += 1
            continue

        for chunk_num, chunk in enumerate(chunks):
            try:
                mel_spec = audio_to_mel_spectrogram(chunk)
                mel_resized = resize_spectrogram(mel_spec)
            except Exception:
                continue
            all_specs.append(mel_resized)
            all_labels.append(label)
            map_rows.append({
                "chunk_index": chunk_idx,
                "track_id": track_id,
                "chunk_number": chunk_num,
                "label": label,
            })
            chunk_idx += 1

    X = np.array(all_specs, dtype=np.float16)[..., np.newaxis]
    y = np.array(all_labels, dtype=np.int32)
    pd.DataFrame(map_rows).to_csv(out_map, index=False)
    np.save(out_X, X)
    np.save(out_y, y)

    print(f"[{split_name}] saved: X={X.shape}, y={y.shape}, skipped={skipped} tracks")


# Preprocess all three splits
for split in ["training", "validation", "test"]:
    preprocess_split(split, metadata)

print("\nAll splits preprocessed.")

In [ ]:
# Load preprocessed data
X_train = np.load(os.path.join(SPECTROGRAM_DIR, "X_training.npy"))
y_train = np.load(os.path.join(SPECTROGRAM_DIR, "y_training.npy"))
X_val = np.load(os.path.join(SPECTROGRAM_DIR, "X_validation.npy"))
y_val = np.load(os.path.join(SPECTROGRAM_DIR, "y_validation.npy"))
X_test = np.load(os.path.join(SPECTROGRAM_DIR, "X_test.npy"))
y_test = np.load(os.path.join(SPECTROGRAM_DIR, "y_test.npy"))
test_map = pd.read_csv(os.path.join(SPECTROGRAM_DIR, "map_test.csv"))

y_train_oh = to_categorical(y_train, NUM_CLASSES)
y_val_oh = to_categorical(y_val, NUM_CLASSES)
y_test_oh = to_categorical(y_test, NUM_CLASSES)

print(f"Train:      X={X_train.shape}, y={y_train.shape}")
print(f"Validation: X={X_val.shape}, y={y_val.shape}")
print(f"Test:       X={X_test.shape}, y={y_test.shape}")
print(f"\nMemory: train={X_train.nbytes / 1e9:.1f} GB, val={X_val.nbytes / 1e9:.1f} GB, test={X_test.nbytes / 1e9:.1f} GB")

In [ ]:
# Visualize sample spectrograms from each genre
n_genres = NUM_CLASSES
n_cols = 4
n_rows = (n_genres + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
axes_flat = axes.flat if n_rows > 1 else [axes] if n_cols == 1 else axes

for i, ax in enumerate(axes_flat):
    if i >= n_genres:
        ax.axis("off")
        continue
    matches = np.where(y_train == i)[0]
    if len(matches) == 0:
        ax.axis("off")
        continue
    idx = matches[0]
    ax.imshow(X_train[idx, :, :, 0].astype(np.float32), aspect="auto", origin="lower", cmap="magma")
    ax.set_title(GENRES[i])
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle(f"Sample Mel Spectrograms (FMA-{SUBSET})", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Build & Train the CNN

Same 5-block CNN as v1 but with stronger regularization (higher dense dropout) since FMA-Medium is harder than GTZAN.

We use **class weights** to handle FMA-Medium's class imbalance — Rock & Electronic dominate, while Spoken & Easy Listening are rare.

In [ ]:
# Compute class weights to handle imbalance
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=y_train,
)
class_weights = {i: float(w) for i, w in enumerate(class_weights_arr)}

print("Class weights (higher = rarer genre):")
for genre, w in sorted(zip(GENRES, class_weights_arr), key=lambda x: -x[1]):
    print(f"  {genre:25s}: {w:.3f}")

In [ ]:
# Build model
tf.keras.utils.set_random_seed(RANDOM_SEED)

inputs = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1))
x = inputs

for filters in CONV_FILTERS:
    x = layers.Conv2D(filters, KERNEL_SIZE, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=POOL_SIZE)(x)
    x = layers.Dropout(CONV_DROPOUT)(x)

x = layers.Flatten()(x)
x = layers.Dense(DENSE_UNITS, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DENSE_DROPOUT)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
# Train
callbacks = [
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

# Cast to float32 only at training time (saves memory storing as float16)
history = model.fit(
    X_train.astype(np.float32), y_train_oh,
    validation_data=(X_val.astype(np.float32), y_val_oh),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weights,
)

# Save training history & genre list for inference
history_dict = {key: [float(v) for v in values] for key, values in history.history.items()}
with open(HISTORY_PATH, "w") as f:
    json.dump(history_dict, f, indent=2)
with open(GENRES_PATH, "w") as f:
    json.dump(GENRES, f, indent=2)

print(f"\nModel saved to: {MODEL_PATH}")
print(f"History saved to: {HISTORY_PATH}")
print(f"Genre list saved to: {GENRES_PATH}")

## 7. Evaluation on Held-Out Test Set

The test set was never seen during training — this is the honest performance number.

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_dict["accuracy"], label="Train Accuracy")
ax1.plot(history_dict["val_accuracy"], label="Val Accuracy")
ax1.set_title("Model Accuracy")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history_dict["loss"], label="Train Loss")
ax2.plot(history_dict["val_loss"], label="Val Loss")
ax2.set_title("Model Loss")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "training_curves_v2.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chunk-level evaluation on test set
y_pred = model.predict(X_test.astype(np.float32), batch_size=BATCH_SIZE, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)

print("=" * 60)
print("CHUNK-LEVEL EVALUATION (held-out test set)")
print("=" * 60)
print(classification_report(y_test, y_pred_classes, target_names=GENRES, zero_division=0))
print(f"Chunk-level accuracy: {accuracy_score(y_test, y_pred_classes):.4f}")

In [ ]:
# File-level evaluation with majority voting
file_preds = []
file_true = []

for track_id, group in test_map.groupby("track_id"):
    indices = group.index.values
    chunk_preds = y_pred_classes[indices]
    majority = Counter(chunk_preds).most_common(1)[0][0]
    file_preds.append(majority)
    file_true.append(group["label"].iloc[0])

file_preds = np.array(file_preds)
file_true = np.array(file_true)

print("=" * 60)
print("FILE-LEVEL EVALUATION (majority voting, held-out test set)")
print("=" * 60)
print(classification_report(file_true, file_preds, target_names=GENRES, zero_division=0))
print(f"File-level accuracy: {accuracy_score(file_true, file_preds):.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(file_true, file_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=GENRES, yticklabels=GENRES)
plt.title("Confusion Matrix (Test Set, File-Level Majority Voting)")
plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "confusion_matrix_v2.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Inference — Top-3 Genre Prediction

Upload a `.wav` or `.mp3` file (full Hollywood/Bollywood/any song works — we chunk it automatically). Returns top 3 genres with probabilities.

**Tip:** Test with songs that the model has never seen — anything from your music library or downloaded from YouTube.

In [ ]:
def predict_genre(file_path, model, top_k=3):
    """
    Predict genre of an audio file.

    Returns dict with:
      - top_predictions: list of (genre, probability), sorted by probability
      - genre_probabilities: dict of all genre -> probability (averaged across chunks)
      - chunk_predictions: per-chunk predicted genre
      - num_chunks: number of chunks processed
    """
    X = preprocess_single_file(file_path, max_chunks=None)
    if len(X) == 0:
        return {"error": "File too short to extract any 4-second chunks"}

    predictions = model.predict(X, verbose=0)
    avg_probs = predictions.mean(axis=0)

    # Top-K
    top_indices = np.argsort(avg_probs)[::-1][:top_k]
    top_predictions = [(GENRES[i], float(avg_probs[i])) for i in top_indices]

    # Per-chunk predictions
    chunk_preds = [GENRES[i] for i in np.argmax(predictions, axis=1)]

    return {
        "top_predictions": top_predictions,
        "genre_probabilities": {g: float(p) for g, p in zip(GENRES, avg_probs)},
        "chunk_predictions": chunk_preds,
        "num_chunks": len(X),
    }

In [ ]:
# Upload and predict (on Colab)
try:
    from google.colab import files
    print("Upload a .wav or .mp3 file:")
    uploaded = files.upload()
    if uploaded:
        audio_path = list(uploaded.keys())[0]
        result = predict_genre(audio_path, model, top_k=3)

        if "error" in result:
            print(f"Error: {result['error']}")
        else:
            print(f"\nProcessed {result['num_chunks']} chunks (~{result['num_chunks'] * 4} seconds)")
            print("\nTop 3 Genres:")
            for rank, (genre, prob) in enumerate(result["top_predictions"], 1):
                print(f"  {rank}. {genre:25s}: {prob:.1%}")

            print("\nAll genre probabilities:")
            for genre, prob in sorted(result["genre_probabilities"].items(), key=lambda x: -x[1]):
                bar = "█" * int(prob * 40)
                print(f"  {genre:25s}: {prob:.3f} {bar}")
except ImportError:
    # Not running on Colab — provide a path manually
    # audio_path = "path/to/your/song.mp3"
    # result = predict_genre(audio_path, model, top_k=3)
    print("Not running on Colab. Set audio_path manually and call predict_genre(audio_path, model).")

## 9. Download Trained Model

Download the model and artifacts to your local machine for the Streamlit app.

In [ ]:
try:
    from google.colab import files
    files.download(MODEL_PATH)
    files.download(HISTORY_PATH)
    files.download(GENRES_PATH)
    files.download(os.path.join(PLOTS_DIR, "training_curves_v2.png"))
    files.download(os.path.join(PLOTS_DIR, "confusion_matrix_v2.png"))
except ImportError:
    print(f"Model:    {MODEL_PATH}")
    print(f"History:  {HISTORY_PATH}")
    print(f"Genres:   {GENRES_PATH}")
    print(f"Plots:    {PLOTS_DIR}")